# TP02 — Comparer et tracer des expérimentations Machine Learning

Ce notebook complète l'exploration de `ElectricityLoadDiagrams20112014` et suit le fil du TP02 :

**dataset → drop NaN → split temporel → stratégies de features → fit → predict → métriques → MLflow → comparaison → Ridge / hyperparamètres**

Principes importants :

- on respecte l'ordre temporel ;
- le jeu de **validation** sert à choisir stratégie et hyperparamètres ;
- le jeu de **test** sert à l'évaluation finale ;
- les expériences sont tracées dans **MLflow** ;
- les versions de splits peuvent être suivies avec **DVC** ;
- un mode `SMOKE_TEST` permet de tester rapidement le code sans saturer la RAM.


## 1. Imports et configuration


In [ ]:
from pathlib import Path
import gc
import json
import os
import subprocess

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = Path("data/modelling")
FEATURES_PATH = DATA_DIR / "features.parquet"
TARGET_PATH = DATA_DIR / "target.parquet"
SPLITS_DIR = Path("data/splits")

TARGET = "consumption_kwh"
RANDOM_STATE = 42

# True = validation rapide du notebook sur un sous-échantillon.
# Pour les runs finaux du TP, repasser à False.
SMOKE_TEST = True
SMOKE_TRAIN_ROWS = 600_000
SMOKE_EVAL_ROWS = 300_000

MLFLOW_EXPERIMENT = "electricity-load-tp02"

print("MLflow tracking URI :", mlflow.get_tracking_uri())
mlflow.set_experiment(MLFLOW_EXPERIMENT)


## 2. Charger les features et la cible

Le TP02 annonce les features suivantes :

- `lag_1d`
- `lag_2d`
- `lag_7d`
- `lag_14d`
- `rolling_mean_7d`
- `rolling_mean_30d`

Le jeu de données actuellement utilisé dans les notebooks précédents peut différer.
Cette cellule le vérifie explicitement au lieu de supposer que les colonnes sont identiques.


In [ ]:
df_features = pq.read_table(FEATURES_PATH).to_pandas()
df_target = pq.read_table(TARGET_PATH).to_pandas()

# Réduction mémoire : les consommations/features n'ont pas besoin de float64 ici.
for col in df_features.select_dtypes(include="number").columns:
    df_features[col] = df_features[col].astype("float32")

df_target[TARGET] = df_target[TARGET].astype("float32")

print("Features :", df_features.shape)
print("Target   :", df_target.shape)
print("Colonnes disponibles :", list(df_features.columns))

EXPECTED_TP_FEATURES = [
    "lag_1d",
    "lag_2d",
    "lag_7d",
    "lag_14d",
    "rolling_mean_7d",
    "rolling_mean_30d",
]

missing = [c for c in EXPECTED_TP_FEATURES if c not in df_features.columns]

if missing:
    print("\n⚠️ Features demandées par le TP absentes du parquet :", missing)
    print("Le notebook utilisera des stratégies compatibles avec les colonnes disponibles.")
    print("Pour coller STRICTEMENT à l'énoncé, il faudra régénérer ces features.")
else:
    print("\n✅ Toutes les features du TP02 sont présentes.")


## 3. Fusion et `dropna()`

Les valeurs `NaN` dues aux lags/rolling windows ne sont pas exploitables par les modèles linéaires.
On les retire **avant** de construire les splits.

> Attention : avec un lag très long (ex. `lag_365d`), une grande partie de 2011 peut disparaître après `dropna()`.


In [ ]:
df = df_features.join(df_target, how="inner")
del df_features, df_target
gc.collect()

print("Avant dropna :", df.shape)
df = df.dropna()
print("Après dropna :", df.shape)

years = df.index.get_level_values("timestamp").year
print("\nObservations après dropna par année :")
print(pd.Series(years).value_counts().sort_index())


## 4. Stratégies de features

On définit plusieurs hypothèses.

Si les colonnes exactes du TP sont disponibles :

1. **court terme** : passé très récent ;
2. **cycle hebdomadaire** : 1, 7 et 14 jours ;
3. **tendance + saisonnalité** : lags + moyennes glissantes.

Sinon, on construit des stratégies équivalentes avec les colonnes réellement présentes.


In [ ]:
available = set(df.columns) - {TARGET}

if set(EXPECTED_TP_FEATURES).issubset(available):
    FEATURE_STRATEGIES = {
        "short_term": {
            "features": ["lag_1d", "lag_2d"],
            "hypothesis": "La consommation dépend surtout des jours immédiatement précédents.",
        },
        "weekly_cycle": {
            "features": ["lag_1d", "lag_7d", "lag_14d"],
            "hypothesis": "Le cycle hebdomadaire est fortement prédictif.",
        },
        "trend_and_seasonality": {
            "features": EXPECTED_TP_FEATURES,
            "hypothesis": "Combiner mémoire courte, cycle hebdomadaire et tendance améliore la prédiction.",
        },
    }
else:
    def existing(*cols):
        return [c for c in cols if c in available]

    FEATURE_STRATEGIES = {
        "short_memory": {
            "features": existing("lag_1d", "lag_7d"),
            "hypothesis": "Une petite quantité d'historique suffit peut-être.",
        },
        "medium_memory": {
            "features": existing("lag_1d", "lag_7d", "lag_30d"),
            "hypothesis": "Ajouter une mémoire mensuelle peut capter une tendance plus lente.",
        },
        "full_available": {
            "features": [c for c in df.columns if c != TARGET],
            "hypothesis": "Toutes les features disponibles peuvent apporter un signal complémentaire.",
        },
    }

FEATURE_STRATEGIES


## 5. Splits temporels

### Split v1 — demandé en Partie 1
- train : 2011–2012
- validation : 2013
- test : 2014

La commande suggérée par le formateur est utilisée via `np.isin(...)`.

### Split v2 — Partie 2
- train : 2013
- validation : début 2014
- test : fin 2014

L'énoncé ne donne pas de date précise pour « début/fin 2014 ».
Ici on pose explicitement l'hypothèse **1er juillet 2014** comme frontière.


In [ ]:
def build_split_v1(frame):
    years = frame.index.get_level_values("timestamp").year

    train_mask = np.isin(years, [2011, 2012])
    valid_mask = years == 2013
    test_mask = years == 2014

    return {
        "name": "split_v1_2011-2012_2013_2014",
        "description": "train=2011-2012 | validation=2013 | test=2014",
        "train": frame.loc[train_mask],
        "validation": frame.loc[valid_mask],
        "test": frame.loc[test_mask],
    }


def build_split_v2(frame, cutoff="2014-07-01"):
    ts = frame.index.get_level_values("timestamp")
    years = ts.year
    cutoff = pd.Timestamp(cutoff)

    train_mask = years == 2013
    valid_mask = (ts >= pd.Timestamp("2014-01-01")) & (ts < cutoff)
    test_mask = ts >= cutoff

    return {
        "name": "split_v2_2013_2014H1_2014H2",
        "description": f"train=2013 | validation=2014-01-01..{cutoff.date()} | test={cutoff.date()}..fin-2014",
        "train": frame.loc[train_mask],
        "validation": frame.loc[valid_mask],
        "test": frame.loc[test_mask],
    }


split_v1 = build_split_v1(df)
split_v2 = build_split_v2(df)

for split in (split_v1, split_v2):
    print("\n", split["description"])
    for part in ("train", "validation", "test"):
        print(f"  {part:10s}: {len(split[part]):,} lignes")


## 6. Option DVC — matérialiser et versionner les splits

Le TP demande que le code traduise réellement le partage des données.

La fonction ci-dessous peut écrire les trois partitions en Parquet.
Elle est volontairement **désactivée par défaut** car cela peut consommer plusieurs Go sur la VM.

Après écriture d'un split :

```bash
dvc add data/splits/split_v1_2011-2012_2013_2014
git add data/splits/split_v1_2011-2012_2013_2014.dvc data/splits/.gitignore
git commit -m "data: version temporal split v1"
```

Même principe pour le split v2.


In [ ]:
WRITE_DVC_SPLITS = False

def materialize_split(split):
    out_dir = SPLITS_DIR / split["name"]
    out_dir.mkdir(parents=True, exist_ok=True)

    for part in ("train", "validation", "test"):
        split[part].to_parquet(
            out_dir / f"{part}.parquet",
            compression="zstd",
        )

    manifest = {
        "name": split["name"],
        "description": split["description"],
        "rows": {part: len(split[part]) for part in ("train", "validation", "test")},
        "columns": list(split["train"].columns),
    }
    (out_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return out_dir

if WRITE_DVC_SPLITS:
    print("Écrit :", materialize_split(split_v1))
    print("Écrit :", materialize_split(split_v2))
else:
    print("WRITE_DVC_SPLITS=False : aucun gros fichier dupliqué pour le moment.")


## 7. Helpers : sous-échantillonnage de smoke test et métriques

Le `SMOKE_TEST` sert uniquement à vérifier que le pipeline fonctionne.

Pour les résultats à rendre, utiliser le dataset complet (`SMOKE_TEST=False`).


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

def maybe_sample(frame, max_rows):
    if not SMOKE_TEST or len(frame) <= max_rows:
        return frame
    idx = rng.choice(len(frame), size=max_rows, replace=False)
    return frame.iloc[idx]


def regression_metrics(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def coefficient_dict(model, feature_names):
    if not hasattr(model, "coef_"):
        return {}
    return {
        feature: float(coef)
        for feature, coef in zip(feature_names, np.ravel(model.coef_))
    }


## 8. Baseline simple

Avant d'affirmer qu'un modèle « apprend bien », on peut le comparer à une référence naïve.

`DummyRegressor(strategy="mean")` ne cherche aucun motif temporel : il prédit la moyenne.


In [ ]:
def run_one_experiment(
    split,
    strategy_name,
    feature_names,
    estimator,
    model_type,
    extra_params=None,
):
    train_df = maybe_sample(split["train"], SMOKE_TRAIN_ROWS)
    valid_df = maybe_sample(split["validation"], SMOKE_EVAL_ROWS)

    X_train = train_df[feature_names]
    y_train = train_df[TARGET]
    X_valid = valid_df[feature_names]
    y_valid = valid_df[TARGET]

    model = clone(estimator)

    with mlflow.start_run(run_name=f"{split['name']}__{strategy_name}__{model_type}"):
        mlflow.log_param("model_type", model_type)
        mlflow.log_param("strategy_name", strategy_name)
        mlflow.log_param("features", ",".join(feature_names))
        mlflow.log_param("split_strategy", split["description"])
        mlflow.log_param("smoke_test", SMOKE_TEST)
        mlflow.log_param("train_rows", len(train_df))
        mlflow.log_param("validation_rows", len(valid_df))

        if extra_params:
            mlflow.log_params(extra_params)

        model.fit(X_train, y_train)
        pred = model.predict(X_valid)

        metrics = regression_metrics(y_valid, pred)
        mlflow.log_metrics({f"val_{k}": v for k, v in metrics.items()})

        coefs = coefficient_dict(model, feature_names)
        if coefs:
            mlflow.log_dict(coefs, "coefficients.json")

        mlflow.sklearn.log_model(
            model,
            artifact_path="model",
            input_example=X_valid.head(5),
        )

        run_id = mlflow.active_run().info.run_id

    gc.collect()

    return {
        "run_id": run_id,
        "split": split["name"],
        "strategy": strategy_name,
        "model_type": model_type,
        "features": feature_names,
        **{f"val_{k}": v for k, v in metrics.items()},
    }


## 9. Partie 1 — Régression linéaire + stratégies de features

Chaque stratégie est entraînée et évaluée sur le split v1.


In [ ]:
part1_results = []

for strategy_name, spec in FEATURE_STRATEGIES.items():
    if not spec["features"]:
        continue

    result = run_one_experiment(
        split=split_v1,
        strategy_name=strategy_name,
        feature_names=spec["features"],
        estimator=LinearRegression(),
        model_type="linear_regression",
        extra_params={"hypothesis": spec["hypothesis"]},
    )
    part1_results.append(result)

part1_df = (
    pd.DataFrame(part1_results)
    .sort_values("val_rmse")
    .reset_index(drop=True)
)

part1_df


## 10. Partie 2 — Comparaison du deuxième split

On rejoue les mêmes stratégies avec des données plus récentes.


In [ ]:
part2_results = []

for strategy_name, spec in FEATURE_STRATEGIES.items():
    if not spec["features"]:
        continue

    result = run_one_experiment(
        split=split_v2,
        strategy_name=strategy_name,
        feature_names=spec["features"],
        estimator=LinearRegression(),
        model_type="linear_regression",
        extra_params={"hypothesis": spec["hypothesis"]},
    )
    part2_results.append(result)

part2_df = (
    pd.DataFrame(part2_results)
    .sort_values("val_rmse")
    .reset_index(drop=True)
)

pd.concat(
    {
        "split_v1": part1_df,
        "split_v2": part2_df,
    },
    names=["experiment_group"],
)


## 11. Partie 3 — Ridge et influence de `alpha`

Le meilleur feature set du split v1 est repris.

Valeurs imposées par le TP :

- `1`
- `1e3`
- `1e9`

Plus `alpha` augmente, plus Ridge pénalise les coefficients élevés.


In [ ]:
best_strategy_name = part1_df.loc[0, "strategy"]
best_features = FEATURE_STRATEGIES[best_strategy_name]["features"]

print("Meilleure stratégie (validation split v1) :", best_strategy_name)
print("Features :", best_features)

ridge_results = []

for alpha in [1.0, 1e3, 1e9]:
    result = run_one_experiment(
        split=split_v1,
        strategy_name=best_strategy_name,
        feature_names=best_features,
        estimator=Ridge(alpha=alpha),
        model_type="ridge",
        extra_params={"alpha": alpha},
    )
    ridge_results.append(result)

ridge_df = (
    pd.DataFrame(ridge_results)
    .sort_values("val_rmse")
    .reset_index(drop=True)
)

ridge_df


## 12. Choix final puis test 2014

Le **test** ne doit pas servir à choisir `alpha`.

On choisit d'abord le meilleur `alpha` grâce à la validation 2013, puis on peut :

1. réentraîner sur train + validation ;
2. mesurer une seule fois la généralisation sur le test 2014.


In [ ]:
best_alpha = float(ridge_df.loc[0, "run_id"] and 0)  # valeur remplacée ci-dessous via MLflow/results
best_ridge_row = ridge_df.iloc[0]

# Retrouver alpha via le run MLflow permet de ne pas dépendre d'une colonne locale non loggée.
client = mlflow.tracking.MlflowClient()
best_run = client.get_run(best_ridge_row["run_id"])
best_alpha = float(best_run.data.params["alpha"])

train_valid = pd.concat([split_v1["train"], split_v1["validation"]])
train_valid = maybe_sample(train_valid, SMOKE_TRAIN_ROWS)
test_df = maybe_sample(split_v1["test"], SMOKE_EVAL_ROWS)

final_model = Ridge(alpha=best_alpha)
final_model.fit(train_valid[best_features], train_valid[TARGET])

test_pred = final_model.predict(test_df[best_features])
test_metrics = regression_metrics(test_df[TARGET], test_pred)

print("alpha choisi :", best_alpha)
print("Test 2014 :", test_metrics)


## 13. Aller plus loin — performances par client

Une métrique globale peut masquer des clients très mal prédits.

On calcule donc la MAE par `individual` sur le test.


In [ ]:
residuals = pd.DataFrame(
    {
        "y_true": test_df[TARGET].to_numpy(),
        "y_pred": test_pred,
    },
    index=test_df.index,
)

residuals["abs_error"] = np.abs(residuals["y_true"] - residuals["y_pred"])

mae_by_client = (
    residuals.groupby(level="individual")["abs_error"]
    .mean()
    .sort_values(ascending=False)
)

print("10 clients avec la MAE la plus élevée :")
mae_by_client.head(10)


## 14. Aller plus loin — autre famille de modèle (bonus)

Pour éviter une Random Forest très coûteuse en RAM sur ~18 millions de lignes,
on peut tester `HistGradientBoostingRegressor` sur un sous-échantillon.

Ce bonus n'est pas requis par le TP02.


In [ ]:
RUN_BONUS_MODEL = False

if RUN_BONUS_MODEL:
    bonus_result = run_one_experiment(
        split=split_v1,
        strategy_name=best_strategy_name,
        feature_names=best_features,
        estimator=HistGradientBoostingRegressor(
            learning_rate=0.08,
            max_iter=150,
            max_leaf_nodes=31,
            random_state=RANDOM_STATE,
        ),
        model_type="hist_gradient_boosting",
        extra_params={
            "learning_rate": 0.08,
            "max_iter": 150,
            "max_leaf_nodes": 31,
        },
    )
    display(pd.DataFrame([bonus_result]))
else:
    print("Bonus désactivé. Passer RUN_BONUS_MODEL=True pour le lancer.")


## 15. Contrôles importants avant le rendu

### Fuite de données
Vérifier comment les rolling means ont été créées.

Pour prédire `y(t)`, une rolling mean ne doit pas utiliser `y(t)` ni le futur.
Une construction sûre ressemble à :

```python
series.shift(1).rolling(window).mean()
```

### Test ≠ validation
- validation : choix des features / hyperparamètres ;
- test : mesure finale, après les choix.

### DVC / Git
Le code du split est versionné avec Git.
Les fichiers de données/splits sont versionnés avec DVC.

### MLflow
Chaque run doit permettre de retrouver :
- stratégie ;
- features ;
- split ;
- modèle ;
- hyperparamètres ;
- RMSE / MAE ;
- coefficients ;
- artifact du modèle.


## 16. Schéma mental du TP02

```text
DATASET
   ↓
dropna()
   ↓
SPLIT TEMPOREL VERSIONNÉ (DVC)
   ├── TRAIN
   │      ↓
   │    .fit()
   │      ↓
   │   MODÈLE
   │
   ├── VALIDATION
   │      ↓
   │ .predict()
   │      ↓
   │ RMSE / MAE
   │      ↓
   │ choix features / alpha
   │
   └── TEST
          ↓
      évaluation finale

Toutes les expériences
          ↓
        MLFLOW
```
